# Flujo SVM

### Creación de vocabulario de bolsa de palabras visuales (BoVW)
Ahora, crearemos un vocabulario visual agrupando los descriptores SIFT mediante K-Means

In [ ]:
from sklearn.cluster import MiniBatchKMeans

#definimos el numero de palabras visuales (clusters)
k = 1000

if all_descriptors_np.shape[0] > 0:
   # Usamos MiniBatchKMeans para mayor eficiencia con grandes conjuntos de datos
   # Es una alternativa más rápida a KMeans
    kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, n_init='auto', verbose=False)
    kmeans.fit(all_descriptors_np)
    visual_vocabulary = kmeans.cluster_centers_
    print(f"Dimensiones del vocabulario visual: {visual_vocabulary.shape}")
else:
    print("No se pudo crear vocabulario,sin descriptores.")
    visual_vocabulary = None

Dimensiones del vocabulario visual: (1000, 128)


### Generación de vectores de características BoVW
Tras crear el vocabulario, representaremos cada imagen como un histograma de estas palabras visuales. Este histograma cuenta cuántas veces aparece cada palabra visual en una imagen.

In [ ]:
if visual_vocabulary is not None:
    # Inicializar BOWImgDescriptorExtractor
    # Utiliza el detector SIFT y el vocabulario KMeans para generar histogramas
    bow_extractor = cv2.BOWImgDescriptorExtractor(sift, cv2.BFMatcher(cv2.NORM_L2))
    bow_extractor.setVocabulary(visual_vocabulary)

    # Generar vectores de características BoVW para cada imagen
    bovw_features = []
    for img_gray in X_smoothed:
        keypoints = sift.detect(img_gray, None)
        # Calcula el histograma BoVW para la imagen
        if keypoints:
            features = bow_extractor.compute(img_gray, keypoints)
            if features is not None:
                bovw_features.append(features.flatten())
            else:
                # Si el cálculo devuelve None, agregue un vector cero
                bovw_features.append(np.zeros(k))
        else:
            #Si no se detectan puntos clave, agregue un vector cero.
            bovw_features.append(np.zeros(k))

    bovw_features_np = np.array(bovw_features)
    print(f"Características BoVW generadas con dimension: {bovw_features_np.shape}")
else:
    bovw_features_np = None
    print("No se pueden generar las características de BoVW, falta el vocabulario.")

Características BoVW generadas con dimension: (30000, 1000)


### Análisis de Componentes Principales (PCA)
Finalmente, aplicaremos PCA para reducir la dimensionalidad de los vectores de características BoVW, lo que puede ayudar a mejorar el rendimiento del modelo y reducir el tiempo de cálculo, especialmente si el espacio de características es muy grande

In [ ]:
from sklearn.decomposition import PCA

if bovw_features_np is not None and bovw_features_np.shape[0] > 0:
  # Inicializar el PCA, conservando el 95% de la varianza
  # También se puede especificar un número fijo de componentes, por ejemplo, n_components=50
    pca = PCA(n_components=0.95, random_state=42)

    #Adaptar PCA a las características de BoVW y transformarlas
    bovw_features_pca = pca.fit_transform(bovw_features_np)

    print(f"características BoVW después de PCA: {bovw_features_pca.shape}")
    print(f"número de componentes seleccionados por PCA: {pca.n_components_}")
else:
    bovw_features_pca = None
    print("No se pudo aplicar PCA. Sin características BoVW .")

características BoVW después de PCA: (30000, 667)
número de componentes seleccionados por PCA: 667


#SVM: sin paralelización
incluye la separacion de los conjuntos sobre train.csv, el escalado, el entrenamiento y la evaluacion

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Separar entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    bovw_features_pca,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

# Escalado
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Entrenar SVM
svm = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    random_state=42
)

print("Entrenando SVM...")

inicio = time.time()
svm.fit(X_train, y_train)

fin =time.time()

t_seq = fin - inicio

# Predicciones
y_pred = svm.predict(X_test)

# Métricas
accuracy = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.4f}\n")

print("Reporte de clasificación:")
print(classification_report(y_test, y_pred))

print("Matriz de confusión:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

Train: (24000, 667)
Test: (6000, 667)
Entrenando SVM...

Accuracy: 0.7430

Reporte de clasificación:
              precision    recall  f1-score   support

           0       1.00      0.73      0.84        33
           1       0.90      0.88      0.89       339
           2       0.91      0.85      0.88       342
           3       0.74      0.69      0.72       215
           4       0.39      0.81      0.53       306
           5       0.61      0.71      0.66       286
           6       0.90      0.56      0.69        62
           7       0.86      0.71      0.78       214
           8       0.70      0.54      0.61       215
           9       0.95      0.82      0.88       227
          10       0.62      0.67      0.64       306
          11       0.93      0.81      0.87       200
          12       0.65      0.75      0.70       321
          13       0.66      0.82      0.73       331
          14       1.00      0.96      0.98       116
          15       0.76      0.55 

# SVM: con paralelización


In [ ]:
# Paralelo
def train_subset(data):
    X_sub, y_sub = data
    model = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    random_state=42
    )
    model.fit(X_sub, y_sub)
    return model

tiempos = []
accura = []
speedups = []
eficiencias = []
karp_flatts = []


NUM_PROCESOS = 7

for i in range(2, NUM_PROCESOS + 1):
  # dividir dataset
  indices = np.array_split(np.arange(len(X_train)), i)
  data_splits = [(X_train[idx], y_train[idx]) for idx in indices]

  start_par = time.time()

  with mp.Pool(processes=i) as pool:
      models = pool.map(train_subset, data_splits)

  t_par = time.time() - start_par
  tiempos.append(t_par)


  p = i
  print(f" procesadores: {p}")

  #1 Speed-up
  sp = t_seq / t_par
  speedups.append(sp)

  #2 Eficiencia
  ef = (sp / p) * 100
  eficiencias.append(ef)


  print(f"Tiempo Paralelo: {t_par:.4f} seg")
  print(f"Speed-up: {sp:.2f}")
  print(f"Eficiencia: {ef:.2f}%")

acc_ind = []
for _, model in enumerate(models):
    pred_par = model.predict(X_test)
    acc_par = accuracy_score(y_test, pred_par)
    acc_ind.append(acc_par)

acc_par = np.mean(acc_ind)

tiempo_prom = np.mean(tiempos)

accura.append(acc_par)


 procesadores: 2
Tiempo Paralelo: 162.7071 seg
Speed-up: 1.27
Eficiencia: 63.73%
Fracción secuencial (Karp-Flatt): 0.5690

 procesadores: 3
Tiempo Paralelo: 117.9908 seg
Speed-up: 1.76
Eficiencia: 58.59%
Fracción secuencial (Karp-Flatt): 0.3534

 procesadores: 4
Tiempo Paralelo: 92.8653 seg
Speed-up: 2.23
Eficiencia: 55.83%
Fracción secuencial (Karp-Flatt): 0.2637

 procesadores: 5
Tiempo Paralelo: 96.9736 seg
Speed-up: 2.14
Eficiencia: 42.77%
Fracción secuencial (Karp-Flatt): 0.3345

 procesadores: 6
Tiempo Paralelo: 74.1809 seg
Speed-up: 2.80
Eficiencia: 46.60%
Fracción secuencial (Karp-Flatt): 0.2292

 procesadores: 7
Tiempo Paralelo: 55.4752 seg
Speed-up: 3.74
Eficiencia: 53.41%
Fracción secuencial (Karp-Flatt): 0.1454



In [ ]:
#print(f"Tiempo secuencial: {t_seq:.2f} segundos")

print("Tiempos paralelos", tiempos)
print(f"Tiempo promedio {tiempo_prom:.7f}s")
print(f"Accuracy promedio: {acc_par:.7f}")
print(f"Tiempo secuecial: {t_seq}")
print("Reporte de clasificación:")
print(classification_report(y_test, y_pred))

print("Matriz de confusión:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
#

Tiempos paralelos [162.7070655822754, 117.99077343940735, 92.86533856391907, 96.97357416152954, 74.18086624145508, 55.47515654563904]
Tiempo promedio 100.0321291s
Accuracy promedio: 0.6025952
[0.6085, 0.6026666666666667, 0.598, 0.5981666666666666, 0.6071666666666666, 0.605, 0.5986666666666667]
Tiempo secuecial: 207.39999866485596
Reporte de clasificación:
              precision    recall  f1-score   support

           0       1.00      0.73      0.84        33
           1       0.90      0.88      0.89       339
           2       0.91      0.85      0.88       342
           3       0.74      0.69      0.72       215
           4       0.39      0.81      0.53       306
           5       0.61      0.71      0.66       286
           6       0.90      0.56      0.69        62
           7       0.86      0.71      0.78       214
           8       0.70      0.54      0.61       215
           9       0.95      0.82      0.88       227
          10       0.62      0.67      0.64    